# Serverless LLMs and Agentic AI with Modal – Lesson 4  
## GPUs + CPU/Memory Reservations

In this lesson we build a tiny **“Embedding Microservice”** that turns text into vector embeddings using a small Transformer model.  
We’ll run the same workload in three modes and compare:

1) **CPU (default resources)**  
2) **CPU with reserved `cpu=` and `memory=`**  
3) **GPU (`gpu="A10G"` by default)**

You’ll learn how to:
- Request a GPU for a Modal function (`gpu=...`)
- Reserve minimum CPU cores and memory (`cpu=...`, `memory=...`)
- See the effect on **throughput** (requests/sec) for model inference
- Debug a GPU container with `modal shell` and `nvidia-smi`

> Why embeddings? This pattern shows up everywhere in serverless LLM apps: search, RAG, clustering, deduplication, and routing.


In [ ]:
# =====================================
# Step 0 – Install and check Modal
# =====================================
!pip install modal --quiet
!which modal
!modal --version
print("✅ Modal installed.")

## Step 1 – Verify authentication







In [ ]:
# ============================
# Step 1B – Configure Modal using the CLI (matches docs)
# ============================
# ⚠️ IMPORTANT:
# - Replace the placeholder strings with your real MODAL_TOKEN_ID and MODAL_TOKEN_SECRET.
# - Do NOT commit these values to GitHub or share them.
#
# This cell:
#   1. Stores your token via `modal token set`.
#   2. This writes the Modal config file (e.g. ~/.modal.toml) for you.

#TOKEN_ID = ""        # <-- paste from Modal dashboard
#TOKEN_SECRET = ""  # <-- paste from Modal dashboard


TOKEN_ID = "ak-0BiEREFFHUeutPdAxq7wnO"        # <-- paste from Modal dashboard
TOKEN_SECRET = "as-7lxmF1GOy4ws6bjYAsbRwS"  # <-- paste from Modal dashboard



if "YOUR_TOKEN_ID_HERE" in TOKEN_ID or "YOUR_TOKEN_SECRET_HERE" in TOKEN_SECRET:
    raise ValueError("❌ Please set TOKEN_ID and TOKEN_SECRET before running this cell.")

# Call the Modal CLI to store the token
!modal token set --token-id $TOKEN_ID --token-secret $TOKEN_SECRET

print("✅ Token stored via `modal token set`. You should be authenticated now.")

## Step 2 – What we’re going to build

We’ll generate a Python script called **`lesson4_gpu_resources.py`**. It defines three Modal functions:

- `embed_cpu_default(...)`  
  Uses default Modal resources (good baseline).

- `embed_cpu_reserved(...)`  
  Same logic, but requests guaranteed resources like `cpu=4`, `memory=2048` (MiB).

- `embed_gpu(...)`  
  Runs the exact same model on a GPU.

### The workload
- We load a small Transformer model: `sentence-transformers/all-MiniLM-L6-v2`  
- We encode a batch of sentences repeatedly and measure elapsed time.
- We return a **dict of metrics** (portable; no extra local libs needed to deserialize).

> Heads-up: The first run includes model download + container warmup. Re-running is much faster because Modal caches the image layers.


In [ ]:
%%writefile lesson4_gpu_resources.py
import time
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List

import modal

app = modal.App("lesson4-embedding-microservice-gpu-resources")

# ------------------------------------------------------------
# Image: install libraries INSIDE the container
# ------------------------------------------------------------
image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "torch==2.4.1",
        "transformers==4.44.2",
        "accelerate==0.33.0",
        "safetensors==0.4.5",
    )
)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def _mean_pool(last_hidden_state, attention_mask):
    """Mean-pool token embeddings with attention mask."""
    import torch

    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def _load_model_and_tokenizer():
    """Load tokenizer + model inside the container."""
    from transformers import AutoModel, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
    model.eval()
    return tok, model


def _run_embedding_benchmark(device: str, batch_size: int, repeats: int, max_length: int) -> Dict:
    """Run a simple embedding throughput benchmark and return metrics."""
    import torch

    tok, model = _load_model_and_tokenizer()
    model.to(device)

    # Build a deterministic batch of sentences
    base = "Modal makes serverless ML feel local. "
    texts = [(base * 5) + f"#{i}" for i in range(batch_size)]

    # Tokenize once (tokenization often stays CPU-side in real systems)
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    # Warm-up (important on GPU)
    with torch.no_grad():
        out = model(**enc)
        emb = _mean_pool(out.last_hidden_state, enc["attention_mask"])
        if device.startswith("cuda"):
            torch.cuda.synchronize()

    # Timed loop
    start = time.time()
    with torch.no_grad():
        for _ in range(repeats):
            out = model(**enc)
            emb = _mean_pool(out.last_hidden_state, enc["attention_mask"])
        if device.startswith("cuda"):
            torch.cuda.synchronize()
    elapsed = time.time() - start

    total_items = batch_size * repeats
    items_per_sec = total_items / max(elapsed, 1e-9)

    return {
        "device": device,
        "model": MODEL_NAME,
        "batch_size": batch_size,
        "repeats": repeats,
        "max_length": max_length,
        "elapsed_sec": round(elapsed, 4),
        "items_total": total_items,
        "items_per_sec": round(items_per_sec, 2),
        "embedding_dim": int(emb.shape[-1]),
    }


# ------------------------------------------------------------
# 1) CPU baseline (default resources)
# ------------------------------------------------------------
@app.function(image=image)
def embed_cpu_default(batch_size: int = 32, repeats: int = 10, max_length: int = 128) -> Dict:
    device = "cpu"
    return _run_embedding_benchmark(device, batch_size, repeats, max_length)


# ------------------------------------------------------------
# 2) CPU with reserved resources
# ------------------------------------------------------------
# Reserve minimum resources for predictable performance.
@app.function(image=image, cpu=4, memory=2048)
def embed_cpu_reserved(batch_size: int = 64, repeats: int = 10, max_length: int = 128) -> Dict:
    device = "cpu"
    return _run_embedding_benchmark(device, batch_size, repeats, max_length)


# ------------------------------------------------------------
# 3)) GPU inference
# ------------------------------------------------------------
@app.function(image=image, gpu="A10G")
def embed_gpu(batch_size: int = 256, repeats: int = 20, max_length: int = 128) -> Dict:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return _run_embedding_benchmark(device, batch_size, repeats, max_length)


@app.local_entrypoint()
def lesson4_main(
    batch_size: int = 32,
    repeats: int = 10,
    max_length: int = 128,
    run_gpu: bool = False,
    run_reserved_cpu: bool = False,
):
    """Lesson 4 entrypoint.

    Boolean flags:
      --run-gpu
      --run-reserved-cpu

    Examples:
      modal run lesson4_gpu_resources.py --batch-size 32 --repeats 10
      modal run lesson4_gpu_resources.py --run-reserved-cpu --batch-size 64 --repeats 10
      modal run lesson4_gpu_resources.py --run-gpu --batch-size 256 --repeats 20

    Note:
      Don't write: --run-gpu True
      Just include the flag to set it True.
    """

    print("\n===============================================")
    print("Lesson 4 – GPUs + CPU/Memory Reservations")
    print("===============================================\n")


    if run_gpu:
        print("Running GPU benchmark...\n")
        res = embed_gpu.remote(batch_size=batch_size, repeats=repeats, max_length=max_length)
        print(res)
        return

    if run_reserved_cpu:
        print("Running CPU benchmark with reserved cpu/memory...\n")
        res = embed_cpu_reserved.remote(batch_size=batch_size, repeats=repeats, max_length=max_length)
        print(res)
        return

    print("Running CPU benchmark (default resources)...\n")
    res = embed_cpu_default.remote(batch_size=batch_size, repeats=repeats, max_length=max_length)
    print(res)


## Step 3 – Run the CPU baseline

This is the best “first run” because it avoids GPU availability/cost and helps you verify everything works.

> First run is slower because the container image builds + model downloads.
> Run it a second time to see caching benefits.


In [ ]:
!modal run lesson4_gpu_resources.py --batch-size 32 --repeats 10 --max-length 128

## Step 4 – Run with reserved CPU + memory

Now we request guaranteed resources:
- `cpu=4`
- `memory=2048` (MiB)

This can reduce variance and often improves throughput.


In [ ]:
!modal run lesson4_gpu_resources.py --run-reserved-cpu --batch-size 64 --repeats 10 --max-length 128

## Step 5 – Run on GPU

This runs the same model on a GPU (default: `A10G` in the script).

Try changing the GPU type inside `lesson4_gpu_resources.py`:
- `"L4"`
- `"A100"`
- `"H100"`

Then rerun and compare `items_per_sec`.


In [ ]:
!modal run lesson4_gpu_resources.py --run-gpu --batch-size 256 --repeats 20 --max-length 128

## Step 6 – Debug a GPU container

To open a shell in the container for the GPU function, run in a terminal:

```bash
modal shell lesson4_gpu_resources.py::embed_gpu
```

Inside the container, try:
- `nvidia-smi`
- `python -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"`


In [ ]:
!modal shell lesson4_gpu_resources.py::embed_gpu
